# Multi-Label Image Classification with Missing Labels

This notebook demonstrates the complete pipeline for training and evaluating a multi-label image classification model that handles missing labels.

## Table of Contents
1. [Setup and Imports](#setup)
2. [Data Exploration](#exploration)
3. [Data Loading and Preprocessing](#preprocessing)
4. [Model Architecture](#model)
5. [Training](#training)
6. [Evaluation](#evaluation)
7. [Inference on New Images](#inference)

## 1. Setup and Imports <a id='setup'></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
from tqdm.notebook import tqdm

# Import custom modules
from utils.data import load_dataset, create_dataloaders, get_transforms
from models.model import create_model, MultiLabelLoss
from train import train_model
from evaluate import evaluate_model, compute_metrics

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Data Exploration <a id='exploration'></a>

Let's explore the dataset structure and statistics.

In [ ]:
# Load labels file
labels_df = pd.read_csv('dataset/labels.txt', sep=' ', header=None)
labels_df.columns = ['filename', 'label_1', 'label_2', 'label_3', 'label_4']

print(f"Total number of images: {len(labels_df)}")
print(f"\nFirst few rows:")
labels_df.head(10)

In [ ]:
# Analyze missing values
label_cols = ['label_1', 'label_2', 'label_3', 'label_4']

missing_counts = {}
for col in label_cols:
    missing = (labels_df[col] == 'NA').sum()
    missing_counts[col] = missing
    print(f"{col}: {missing} missing ({missing/len(labels_df)*100:.2f}%)")

# Visualize missing values
plt.figure(figsize=(10, 6))
plt.bar(missing_counts.keys(), missing_counts.values(), color='coral')
plt.xlabel('Label', fontsize=12)
plt.ylabel('Count of Missing Values', fontsize=12)
plt.title('Missing Values per Label', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
for i, (label, count) in enumerate(missing_counts.items()):
    plt.text(i, count + 5, str(count), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze label distribution (excluding NA)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, col in enumerate(label_cols):
    # Convert to numeric, coercing NA to NaN
    numeric_col = pd.to_numeric(labels_df[col], errors='coerce')
    
    # Count available labels
    value_counts = numeric_col.value_counts().sort_index()
    
    axes[i].bar(value_counts.index, value_counts.values, color=['skyblue', 'lightcoral'])
    axes[i].set_xlabel('Label Value', fontsize=11)
    axes[i].set_ylabel('Count', fontsize=11)
    axes[i].set_title(f'{col} Distribution\n(excluding NA)', fontsize=12, fontweight='bold')
    axes[i].set_xticks([0, 1])
    
    # Add value labels on bars
    for j, v in enumerate(value_counts.values):
        axes[i].text(value_counts.index[j], v + 5, str(int(v)), ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize some sample images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

sample_indices = np.random.choice(len(labels_df), 8, replace=False)

for i, idx in enumerate(sample_indices):
    img_path = os.path.join('dataset/images', labels_df.iloc[idx]['filename'])
    img = Image.open(img_path)
    
    axes[i].imshow(img)
    axes[i].axis('off')
    
    # Create label string
    labels = labels_df.iloc[idx][label_cols].values
    label_str = f"Labels: {labels}"
    axes[i].set_title(label_str, fontsize=10)

plt.suptitle('Sample Images with Labels', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Loading and Preprocessing <a id='preprocessing'></a>

In [ ]:
# Load and split dataset
dataset_splits = load_dataset(
    labels_path='dataset/labels.txt',
    train_size=0.7,
    val_size=0.15,
    random_state=42
)

In [ ]:
# Create dataloaders
dataloaders = create_dataloaders(
    dataset_splits,
    batch_size=32,
    num_workers=2,  # Reduce if you have fewer CPU cores
    image_dir='dataset/images'
)

print(f"Train batches: {len(dataloaders['train'])}")
print(f"Val batches: {len(dataloaders['val'])}")
print(f"Test batches: {len(dataloaders['test'])}")

In [ ]:
# Visualize a batch
batch = next(iter(dataloaders['train']))

print(f"Batch images shape: {batch['image'].shape}")
print(f"Batch labels shape: {batch['labels'].shape}")
print(f"Batch masks shape: {batch['mask'].shape}")

# Denormalize and display images
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(8):
    img = batch['image'][i] * std + mean
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img)
    axes[i].axis('off')
    
    labels = batch['labels'][i].numpy()
    mask = batch['mask'][i].numpy()
    
    label_str = "Labels: "
    for j in range(4):
        if mask[j] == 1:
            label_str += f"{int(labels[j])} "
        else:
            label_str += "NA "
    
    axes[i].set_title(label_str, fontsize=10)

plt.suptitle('Batch Sample (After Augmentation)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Model Architecture <a id='model'></a>

In [ ]:
# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = create_model(
    model_type='resnet50',
    num_labels=4,
    pretrained=True
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel: ResNet50")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Test forward pass
dummy_batch = next(iter(dataloaders['train']))
dummy_images = dummy_batch['image'].to(device)

with torch.no_grad():
    outputs = model(dummy_images)

print(f"Input shape: {dummy_images.shape}")
print(f"Output shape: {outputs.shape}")
print(f"Output (logits) sample:\n{outputs[0]}")
print(f"\nOutput (probabilities) sample:\n{torch.sigmoid(outputs[0])}")

## 5. Training <a id='training'></a>

**Note**: For a full training run, use the `train.py` script. This section demonstrates a short training loop.

In [ ]:
# Set up training
criterion = MultiLabelLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# Training parameters
num_epochs = 5  # Small number for demonstration

train_losses = []
val_losses = []

print("Starting training...\n")

In [ ]:
# Training loop
for epoch in range(1, num_epochs + 1):
    # Training
    model.train()
    train_loss = 0.0
    
    pbar = tqdm(dataloaders['train'], desc=f'Epoch {epoch}/{num_epochs} - Train')
    for batch in pbar:
        images = batch['image'].to(device)
        labels = batch['labels'].to(device)
        masks = batch['mask'].to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    train_loss /= len(dataloaders['train'])
    train_losses.append(train_loss)
    
    # Validation
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        pbar = tqdm(dataloaders['val'], desc=f'Epoch {epoch}/{num_epochs} - Val')
        for batch in pbar:
            images = batch['image'].to(device)
            labels = batch['labels'].to(device)
            masks = batch['mask'].to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels, masks)
            
            val_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    val_loss /= len(dataloaders['val'])
    val_losses.append(val_loss)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    print(f"\nEpoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}\n")

print("Training completed!")

In [ ]:
# Plot training history
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_losses, marker='o', label='Train Loss', linewidth=2)
plt.plot(range(1, num_epochs + 1), val_losses, marker='s', label='Val Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Evaluation <a id='evaluation'></a>

In [ ]:
# Evaluate on test set
print("Evaluating on test set...\n")

metrics, label_metrics, predictions, targets, masks = evaluate_model(
    model, dataloaders['test'], device, threshold=0.5
)

print("\n" + "="*60)
print("Test Set Evaluation Results")
print("="*60)

print(f"\nOverall Metrics:")
print(f"  Subset Accuracy: {metrics['subset_accuracy']:.4f}")
print(f"  Hamming Loss: {metrics['hamming_loss']:.4f}")
print(f"  Jaccard Score: {metrics['jaccard_score']:.4f}")
if metrics['roc_auc'] is not None:
    print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")

print(f"\nSample-wise Metrics:")
print(f"  Precision: {metrics['sample_precision']:.4f}")
print(f"  Recall: {metrics['sample_recall']:.4f}")
print(f"  F1 Score: {metrics['sample_f1']:.4f}")

print(f"\nLabel-wise Metrics (Macro Average):")
print(f"  Accuracy: {metrics['macro_accuracy']:.4f}")
print(f"  Precision: {metrics['macro_precision']:.4f}")
print(f"  Recall: {metrics['macro_recall']:.4f}")
print(f"  F1 Score: {metrics['macro_f1']:.4f}")

In [ ]:
# Visualize per-label metrics
labels = ['Label 1', 'Label 2', 'Label 3', 'Label 4']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Per-Label Metrics on Test Set', fontsize=16, fontweight='bold')

# Precision
axes[0, 0].bar(labels, label_metrics['precision'], color='skyblue', edgecolor='navy')
axes[0, 0].set_title('Precision', fontsize=13, fontweight='bold')
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(label_metrics['precision']):
    axes[0, 0].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')

# Recall
axes[0, 1].bar(labels, label_metrics['recall'], color='lightcoral', edgecolor='darkred')
axes[0, 1].set_title('Recall', fontsize=13, fontweight='bold')
axes[0, 1].set_ylim([0, 1])
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(label_metrics['recall']):
    axes[0, 1].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')

# F1 Score
axes[1, 0].bar(labels, label_metrics['f1'], color='lightgreen', edgecolor='darkgreen')
axes[1, 0].set_title('F1 Score', fontsize=13, fontweight='bold')
axes[1, 0].set_ylim([0, 1])
axes[1, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(label_metrics['f1']):
    axes[1, 0].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')

# Accuracy
axes[1, 1].bar(labels, label_metrics['accuracy'], color='plum', edgecolor='purple')
axes[1, 1].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[1, 1].set_ylim([0, 1])
axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(label_metrics['accuracy']):
    axes[1, 1].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 7. Inference on New Images <a id='inference'></a>

In [ ]:
# Function to predict on a single image
def predict_image(model, image_path, transform, device, threshold=0.5):
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        output = model(image_tensor)
        probs = torch.sigmoid(output).cpu().numpy()[0]
    
    # Apply threshold
    predictions = (probs >= threshold).astype(int)
    
    return image, probs, predictions

In [ ]:
# Predict on random test images
transform = get_transforms(augment=False)
test_filenames = dataset_splits['test']['filenames']

# Select random images
sample_indices = np.random.choice(len(test_filenames), 4, replace=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, idx in enumerate(sample_indices):
    img_path = os.path.join('dataset/images', test_filenames[idx])
    image, probs, predictions = predict_image(model, img_path, transform, device)
    
    axes[i].imshow(image)
    axes[i].axis('off')
    
    # Create prediction text
    pred_text = "Predictions:\n"
    for j in range(4):
        pred_text += f"Label {j+1}: {predictions[j]} (prob: {probs[j]:.3f})\n"
    
    axes[i].text(0.5, -0.1, pred_text, transform=axes[i].transAxes, 
                ha='center', va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Sample Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated:
1. ✅ Data exploration and visualization
2. ✅ Handling missing labels in multi-label classification
3. ✅ Building a deep learning model with transfer learning
4. ✅ Training with custom loss function
5. ✅ Comprehensive evaluation with multiple metrics
6. ✅ Making predictions on new images

### Next Steps
- Train for more epochs using `train.py` for better performance
- Experiment with different architectures (ResNet18, EfficientNet, etc.)
- Try different hyperparameters and augmentation strategies
- Analyze failure cases and improve the model